# Pipeline de Machine Learning Reproducible (sklearn + feature-engine)

---

Este notebook construye un pipeline genérico y reproducible acorde al estilo de referencia, para cualquier dataset tabular similar a `data/train.csv`. Todas las listas de variables y mappings están parametrizados como CONSTANTES en mayúsculas al inicio del notebook. Incluye custom transformers, transformaciones dinámicas según el tipo de variable y serialización con joblib.

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import Binarizer, MinMaxScaler
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer, AddMissingIndicator
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.transformation import LogTransformer, YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.selection import DropFeatures
import joblib

## 2. Carga de datos y preprocesamientos iniciales

In [ ]:
data = pd.read_csv('data/train.csv')
data['MSSubClass'] = data['MSSubClass'].astype(str)

---

## 3. Definir CONSTANTES - Listas de variables dinámicamente

In [ ]:
# Variables numéricas y categóricas según análisis del profiler y reglas del dominio
def get_variable_lists(df):
    num_vars = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_vars = df.select_dtypes(include=['object']).columns.tolist()

    # Quitar el target y columnas que no deben ser consideradas
    num_vars.remove('SalePrice')
    if 'Id' in num_vars:
        num_vars.remove('Id')
    if 'Id' in cat_vars:
        cat_vars.remove('Id')

    # Temporal: Año construcción, remodelación y garage
    temporal_vars = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
    reference_var = 'YrSold'

    # Variables con NA numéricas
    numerical_vars_with_na = [var for var in num_vars if df[var].isnull().sum() > 0]
    # Variables categóricas con NA
    categorical_vars_with_na = [var for var in cat_vars if df[var].isnull().sum() > 0]
    # De éstas, las que conceptualmente significan ausencia real ("missing")
    cat_na_missing = [var for var in categorical_vars_with_na if df[var].isnull().mean() > 0.05]
    cat_na_frequent = [var for var in categorical_vars_with_na if var not in cat_na_missing]
    
    return {
        'NUMERICAL_VARS': num_vars,
        'CATEGORICAL_VARS': cat_vars,
        'NUMERICAL_VARS_WITH_NA': numerical_vars_with_na,
        'CATEGORICAL_VARS_WITH_NA_MISSING': cat_na_missing,
        'CATEGORICAL_VARS_WITH_NA_FREQUENT': cat_na_frequent,
        'TEMPORAL_VARS': temporal_vars,
        'REF_VAR': reference_var
    }

# --- Detectar otras variables de transformación especial ---
# Skew alto para log-transform
NUMERICALS_LOG_VARS = ['LotArea','1stFlrSF','GrLivArea','TotalBsmtSF','SalePrice','MiscVal','PoolArea','3SsnPorch','LowQualFinSF','KitchenAbvGr','BsmtFinSF2','ScreenPorch','BsmtHalfBath','EnclosedPorch','MasVnrArea','OpenPorchSF','LotFrontage','BsmtFinSF1','WoodDeckSF','MSSubClass','BsmtUnfSF','2ndFlrSF']

# Excluir target para log-transform y transformar SalePrice antes del fit
def remove_target(var_list):
    return [v for v in var_list if v != 'SalePrice']

# Variables con >50% ceros: candidatas a binarización
BINARIZE_VARS = ['MasVnrArea','BsmtFinSF2','2ndFlrSF','LowQualFinSF','BsmtFullBath','BsmtHalfBath','HalfBath','WoodDeckSF','EnclosedPorch','3SsnPorch','ScreenPorch','PoolArea','MiscVal']

# Candidato Yeo-Johnson: uso para restantes no log-transformables, ej.
NUMERICALS_YEO_VARS = [v for v in get_variable_lists(data)['NUMERICAL_VARS'] if v not in NUMERICALS_LOG_VARS and v not in BINARIZE_VARS and v not in ['YearBuilt','YearRemodAdd','GarageYrBlt']]

# Variables de ordinales de calidad
QUAL_VARS = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
QUAL_MAPPINGS = {
    'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, np.nan: 0
}

EXPOSURE_VARS = ['BsmtExposure']
EXPOSURE_MAPPINGS = {'Gd': 4,'Av': 3,'Mn': 2,'No': 1, np.nan: 0}

FIN_VARS = ['BsmtFinType1','BsmtFinType2']
FIN_MAPPINGS = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, np.nan: 0}

# Otras mappings quedan parametrizables si necesarias (agregar más según reglas de negocio)

# Agregamos también la lista de variables categóricas cuyo encoding será ordinal
CATEGORICAL_VARS = get_variable_lists(data)['CATEGORICAL_VARS']
NUMERICAL_VARS = get_variable_lists(data)['NUMERICAL_VARS']
NUMERICAL_VARS_WITH_NA = get_variable_lists(data)['NUMERICAL_VARS_WITH_NA']
CATEGORICAL_VARS_WITH_NA_MISSING = get_variable_lists(data)['CATEGORICAL_VARS_WITH_NA_MISSING']
CATEGORICAL_VARS_WITH_NA_FREQUENT = get_variable_lists(data)['CATEGORICAL_VARS_WITH_NA_FREQUENT']
TEMPORAL_VARS = get_variable_lists(data)['TEMPORAL_VARS']
REF_VAR = get_variable_lists(data)['REF_VAR']

---

## 4. Definición de custom transformers: TemporalVariableTransformer & Mapper

In [ ]:
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variables, reference_variable):
        if not isinstance(variables, list):
            raise ValueError('variables should be a list')
        self.variables = variables
        self.reference_variable = reference_variable
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for feature in self.variables:
            X[feature] = X[self.reference_variable] - X[feature]
        return X

class Mapper(BaseEstimator, TransformerMixin):
    def __init__(self, variables, mappings):
        if not isinstance(variables, list):
            raise ValueError('variables should be a list')
        self.variables = variables
        self.mappings = mappings
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for feature in self.variables:
            X[feature] = X[feature].map(self.mappings)
        return X

---

## 5. Separación Train/Test y transformación log del objetivo

In [ ]:
X = data.drop(['SalePrice','Id'], axis=1)
y = np.log(data['SalePrice'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=0
)

---

## 6. Construcción del Pipeline

In [ ]:
# ========================== IMPUTATION ==========================
steps = []

if len(CATEGORICAL_VARS_WITH_NA_MISSING) > 0:
    steps.append(('missing_imputation', CategoricalImputer(imputation_method='missing', variables=CATEGORICAL_VARS_WITH_NA_MISSING)))
if len(CATEGORICAL_VARS_WITH_NA_FREQUENT) > 0:
    steps.append(('frequent_imputation', CategoricalImputer(imputation_method='frequent', variables=CATEGORICAL_VARS_WITH_NA_FREQUENT)))
if len(NUMERICAL_VARS_WITH_NA) > 0:
    steps.append(('missing_indicator', AddMissingIndicator(variables=NUMERICAL_VARS_WITH_NA)))
    steps.append(('mean_imputation', MeanMedianImputer(imputation_method='mean', variables=NUMERICAL_VARS_WITH_NA)))

# ======================== TEMPORAL VARIABLES ===================
if len(TEMPORAL_VARS) > 0:
    steps.append(('elapsed_time', TemporalVariableTransformer(variables=TEMPORAL_VARS, reference_variable=REF_VAR)))
    steps.append(('drop_features', DropFeatures(features_to_drop=[REF_VAR])))

# ===================== VARIABLE TRANSFORMATION =================
if len(NUMERICALS_LOG_VARS) > 0:
    steps.append(('log', LogTransformer(variables=remove_target(NUMERICALS_LOG_VARS))))
if len(NUMERICALS_YEO_VARS) > 0:
    steps.append(('yeojohnson', YeoJohnsonTransformer(variables=NUMERICALS_YEO_VARS)))
if len(BINARIZE_VARS) > 0:
    steps.append(('binarizer', SklearnTransformerWrapper(transformer=Binarizer(threshold=0), variables=BINARIZE_VARS)))

# =========================== MAPPERS ===========================
steps.append(('mapper_qual', Mapper(variables=QUAL_VARS, mappings=QUAL_MAPPINGS)))
steps.append(('mapper_exposure', Mapper(variables=EXPOSURE_VARS, mappings=EXPOSURE_MAPPINGS)))
steps.append(('mapper_fin', Mapper(variables=FIN_VARS, mappings=FIN_MAPPINGS)))

# ================== CATEGORICAL ENCODING ======================
steps.append(('rare_label_encoder', RareLabelEncoder(tol=0.01, n_categories=1, variables=CATEGORICAL_VARS)))
steps.append(('categorical_encoder', OrdinalEncoder(encoding_method='ordered', variables=CATEGORICAL_VARS)))

# =============== FEATURE SCALING (Opcional) ===================
# steps.append(('scaler', SklearnTransformerWrapper(transformer=MinMaxScaler(), variables=NUMERICAL_VARS + BINARIZE_VARS)))

pipeline = Pipeline(steps)

---

## 7. Entrenamiento del Pipeline sobre X_train

In [ ]:
pipeline.fit(X_train, y_train)

---

## 8. Transformación de X_train y X_test

In [ ]:
X_train_t = pipeline.transform(X_train)
X_test_t = pipeline.transform(X_test)

---

## 9. Verificación de ausencia de NaN tras la transformación

In [ ]:
assert pd.DataFrame(X_train_t, columns=X_train.columns).isnull().sum().sum() == 0, 'NaNs en el train set transformado!'
assert pd.DataFrame(X_test_t, columns=X_test.columns).isnull().sum().sum() == 0, 'NaNs en el test set transformado!'

---

## 10. Serialización del pipeline

In [ ]:
joblib.dump(pipeline, 'proyecto_pipeline_machine_learning_muestra/preprocessors.pkl')